In [1]:
import pandas as pd

FILE_PATH = r"G:\loan_dataset_investor.csv"

df = pd.read_csv(FILE_PATH, parse_dates=["loan_issued_at"])
print(f"Total rows loaded: {len(df):,}")
print(f"Date range in raw data: {df['loan_issued_at'].min()} to {df['loan_issued_at'].max()}")
df.head()

C:\Users\KUMAR\AppData\Local\Temp\ipykernel_19576\3010299946.py:5: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(FILE_PATH, parse_dates=["loan_issued_at"])


Total rows loaded: 737,889
Date range in raw data: 2009-02-28 18:05:00 to 2026-03-31 02:40:00


,loan_id,country,loan_issued_at,early_repaid_at,is_early_repaid_within_14_days,issued_amount,loan_status,loan_last_recorded_action_date_local,initial_interest_rate,nr_of_payments,...,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,projected_npv_return,customer_risk_rating,Unnamed: 31
0,0B3B642A-F923-49AD-84ED-B2A700BF1D0A,Finland,2025-03-21 11:45:00,04-04-2025 10:50,True,310.0,Repaid,04-04-2025,0.1780,1,...,NaN,0.0,Cancelled,311.99,12,2068.68,0.0,0.018929,F,NaN
1,4FD1927C-0E7F-4BF1-AED0-B268011E3E85,Finland,2025-01-17 17:27:00,NaN,NaN,414.0,Active,NaN,0.1780,120,...,NaN,15.0,Active,186.61,18,1469.50,0.0,0.124216,B,NaN
2,7362E260-54DC-4798-8848-B18C00A59FCB,Finland,2024-06-11 10:04:00,02-01-2025 12:07,False,518.0,Repaid,02-01-2025,0.1877,8,...,NaN,6.0,Paid Up,569.60,60,4513.20,0.0,0.119993,B,NaN
3,8C7E5FB6-A673-4148-BCCF-B27500B0368E,Finland,2025-01-30 10:52:00,NaN,NaN,3731.0,Active,NaN,0.1780,75,...,NaN,15.0,Active,1172.69,60,1301.00,0.0,0.101424,C,NaN
4,BE865034-6041-4855-AB42-B196014FA0E8,Netherlands,2024-06-21 20:48:00,NaN,NaN,3432.0,Active,NaN,0.1056,38,...,NaN,22.0,Active,1670.78,60,5194.70,0.0,0.109355,A,NaN


In [2]:
print(df.columns.tolist())

['loan_id', 'country', 'loan_issued_at', 'early_repaid_at', 'is_early_repaid_within_14_days', 'issued_amount', 'loan_status', 'loan_last_recorded_action_date_local', 'initial_interest_rate', 'nr_of_payments', 'principal_balance', 'principal_debt', 'principal_paid_total', 'interest_paid_total', 'extra_interest_paid_total', 'late_fee_paid_total', 'maintenance_fee_paid_total', 'is_default', 'next_payment_nr', 'next_payment_date_local', 'debt_occured_date_local', 'days_past_due_principal', 'months_in_default', 'months_on_book', 'loan_status_risk', 'repaid_amount_total', 'initial_loan_duration', 'combined_income', 'has_default_within_12_months', 'projected_npv_return', 'customer_risk_rating', 'Unnamed: 31']


In [3]:
# Check the target variable
print(df['has_default_within_12_months'].value_counts(dropna=False))

# Check for that stray empty column
print(df['Unnamed: 31'].isna().sum(), "out of", len(df))

has_default_within_12_months
0.0    480438
NaN    174379
1.0     83072
Name: count, dtype: int64
737889 out of 737889


In [4]:
import pandas as pd

# Drop the empty stray column
df = df.drop(columns=["Unnamed: 31"])

# -----------------------------------------------------
# Define the hard maturity cutoff
# -----------------------------------------------------
DATA_DOWNLOAD_DATE = pd.Timestamp("2026-08-08")
MATURITY_CUTOFF = DATA_DOWNLOAD_DATE - pd.DateOffset(months=12)
print(f"Maturity cutoff date: loans must be issued on or before {MATURITY_CUTOFF.date()}")

# -----------------------------------------------------
# Define the study window: 2018-01-01 through the cutoff
# -----------------------------------------------------
WINDOW_START = pd.Timestamp("2018-01-01")
WINDOW_END = min(pd.Timestamp("2023-12-31"), MATURITY_CUTOFF)
print(f"Final study window: {WINDOW_START.date()} to {WINDOW_END.date()}")

df_filtered = df[
    (df["loan_issued_at"] >= WINDOW_START) &
    (df["loan_issued_at"] <= WINDOW_END)
].copy()

print(f"\nRows after date-window filter: {len(df_filtered):,}")

# Drop any remaining NULL-target rows as a backup check
before = len(df_filtered)
df_filtered = df_filtered[df_filtered["has_default_within_12_months"].notna()].copy()
after = len(df_filtered)
print(f"Rows dropped for missing target after cutoff (should be near 0): {before - after}")

# Sanity checks
default_rate = df_filtered["has_default_within_12_months"].mean()
print(f"\nFinal row count: {len(df_filtered):,}")
print(f"Overall default rate: {default_rate:.2%}")

df_filtered["issue_year"] = df_filtered["loan_issued_at"].dt.year
print("\nRows and default rate by year:")
print(df_filtered.groupby("issue_year").agg(
    n_loans=("loan_id", "count"),
    default_rate=("has_default_within_12_months", "mean")
))

print("\nRows and default rate by country:")
print(df_filtered.groupby("country").agg(
    n_loans=("loan_id", "count"),
    default_rate=("has_default_within_12_months", "mean")
).sort_values("n_loans", ascending=False))

Maturity cutoff date: loans must be issued on or before 2025-08-08
Final study window: 2018-01-01 to 2023-12-31

Rows after date-window filter: 306,470
Rows dropped for missing target after cutoff (should be near 0): 0

Final row count: 306,470
Overall default rate: 16.64%

Rows and default rate by year:
            n_loans  default_rate
issue_year                       
2018          25359      0.268386
2019          56506      0.295508
2020          27874      0.172777
2021          51742      0.102953
2022          63594      0.133692
2023          81395      0.108668

Rows and default rate by country:
             n_loans  default_rate
country                           
Finland       142336      0.152744
Estonia       138040      0.135229
Spain          20645      0.489659
Netherlands     5448      0.087555
Latvia             1      0.000000


In [1]:
import pandas as pd

df = pd.read_csv(r"E:\loan_dataset_2018_2023_filtered.csv", parse_dates=["loan_issued_at"])
print(f"Starting rows: {len(df):,}\n")

# =========================================================
# STEP 4: Duplicate check
# =========================================================
dup_count = df["loan_id"].duplicated().sum()
print(f"STEP 4 - Duplicate loan_id rows: {dup_count}")
if dup_count > 0:
    df = df.drop_duplicates(subset="loan_id", keep="first")
    print(f"  Dropped duplicates. Rows now: {len(df):,}")
else:
    print("  No duplicates found.")

# =========================================================
# STEP 5: Placeholder code check
# =========================================================
print("\nSTEP 5 - Checking for placeholder codes (-1, suspicious 0s):")
for col in ["issued_amount", "initial_interest_rate", "initial_loan_duration"]:
    neg_one = (df[col] == -1).sum()
    zero = (df[col] == 0).sum()
    print(f"  {col}: -1 count = {neg_one}, 0 count = {zero}")

# =========================================================
# STEP 6: Text consistency check
# =========================================================
print("\nSTEP 6 - Country value consistency:")
print(f"  Unique values before cleaning: {sorted(df['country'].unique())}")
df["country"] = df["country"].str.strip().str.title()
print(f"  Unique values after strip+title case: {sorted(df['country'].unique())}")

# =========================================================
# STEP 7: Data type check
# =========================================================
print("\nSTEP 7 - Data types:")
print(df.dtypes)

# =========================================================
# STEP 8: Range sanity check
# =========================================================
print("\nSTEP 8 - Range check on key numeric columns:")
print(df[["issued_amount", "initial_interest_rate", "initial_loan_duration"]].describe())

Starting rows: 306,470

STEP 4 - Duplicate loan_id rows: 0
  No duplicates found.

STEP 5 - Checking for placeholder codes (-1, suspicious 0s):
  issued_amount: -1 count = 0, 0 count = 0
  initial_interest_rate: -1 count = 0, 0 count = 0
  initial_loan_duration: -1 count = 0, 0 count = 0

STEP 6 - Country value consistency:
  Unique values before cleaning: ['Estonia', 'Finland', 'Latvia', 'Netherlands', 'Spain']
  Unique values after strip+title case: ['Estonia', 'Finland', 'Latvia', 'Netherlands', 'Spain']

STEP 7 - Data types:
loan_id                                         object
country                                         object
loan_issued_at                          datetime64[ns]
early_repaid_at                                 object
is_early_repaid_within_14_days                  object
issued_amount                                  float64
loan_status                                     object
loan_last_recorded_action_date_local            object
initial_interest_rate    

In [2]:
print(df["is_default"].unique())
print(df["is_default"].value_counts(dropna=False))

[False True nan]
is_default
False    245336
True      61070
NaN          64
Name: count, dtype: int64


In [3]:
# Confirm these 64 rows still have a valid target
nan_is_default_rows = df[df["is_default"].isna()]
print(nan_is_default_rows["has_default_within_12_months"].value_counts(dropna=False))

has_default_within_12_months
0.0    64
Name: count, dtype: int64


In [4]:
# =========================================================
# STEP 9: Missing values
# =========================================================

# customer_risk_rating: categorical, 92% missing -> drop, keep flag
df["customer_risk_rating_was_missing"] = df["customer_risk_rating"].isna().astype(int)
df = df.drop(columns=["customer_risk_rating"])
print("Dropped customer_risk_rating (92% missing, categorical)")

# combined_income: 100% missing for 2018-2021, structural gap -> drop entirely
df = df.drop(columns=["combined_income"])
print("Dropped combined_income (structural gap: 100% missing 2018-2021)")

# initial_interest_rate: confirmed 0 missing earlier, nothing to do
print(f"initial_interest_rate missing: {df['initial_interest_rate'].isna().sum()}")

# =========================================================
# STEP 10: Outliers (percentile-based, 1st/99th)
# =========================================================
def cap_outliers_percentile(series, lower_pct=0.01, upper_pct=0.99):
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower, upper)

for col in ["issued_amount", "initial_interest_rate"]:
    before_max = df[col].max()
    affected = ((df[col] < df[col].quantile(0.01)) | (df[col] > df[col].quantile(0.99))).sum()
    df[col] = cap_outliers_percentile(df[col])
    print(f"{col}: max {before_max:.2f} -> {df[col].max():.2f}, {affected:,} rows affected ({affected/len(df):.2%})")

print(f"\nRows remaining after Steps 9-10: {len(df):,}")

Dropped customer_risk_rating (92% missing, categorical)
Dropped combined_income (structural gap: 100% missing 2018-2021)
initial_interest_rate missing: 0
issued_amount: max 15948.00 -> 10630.00, 4,971 rows affected (1.62%)
initial_interest_rate: max 1.39 -> 0.60, 4,572 rows affected (1.49%)

Rows remaining after Steps 9-10: 306,470


In [6]:
df["loan_amount_band"] = pd.qcut(
    df["issued_amount"], q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
)

df["loan_term_bucket"] = pd.cut(
    df["initial_loan_duration"],
    bins=[0, 12, 24, 36, 48, 60, 999],
    labels=["<=12mo", "13-24mo", "25-36mo", "37-48mo", "49-60mo", "60mo+"]
)

df["interest_rate_bucket"] = pd.qcut(
    df["initial_interest_rate"], q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
)

country_counts = df["country"].value_counts()
top_countries = country_counts[country_counts >= 100].index.tolist()
df["country_grouped"] = df["country"].where(df["country"].isin(top_countries), "Other")

print("Engineered features created:")
for col in ["loan_amount_band", "loan_term_bucket", "interest_rate_bucket", "country_grouped"]:
    print(f"  {col}: {df[col].nunique()} categories")

Engineered features created:
  loan_amount_band: 5 categories
  loan_term_bucket: 6 categories
  interest_rate_bucket: 5 categories
  country_grouped: 5 categories


In [7]:
LEAKAGE_BLACKLIST = [
    "is_default", "has_default_within_12_months", "days_past_due_principal",
    "months_in_default", "loan_status", "loan_status_risk", "principal_balance",
    "principal_debt", "principal_paid_total", "interest_paid_total",
    "extra_interest_paid_total", "late_fee_paid_total", "maintenance_fee_paid_total",
    "repaid_amount_total", "next_payment_nr", "next_payment_date_local",
    "projected_npv_return", "early_repaid_at", "is_early_repaid_within_14_days",
    "loan_last_recorded_action_date_local", "debt_occured_date_local", "months_on_book",
]
TARGET = "has_default_within_12_months"

candidate_features = [
    "issued_amount", "initial_interest_rate", "initial_loan_duration",
    "loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
    "country_grouped", "customer_risk_rating_was_missing",
]

leaked = [f for f in candidate_features if f in LEAKAGE_BLACKLIST]
if leaked:
    raise ValueError(f"LEAKAGE DETECTED: {leaked}")
print("Leakage check passed.")

Leakage check passed.


In [8]:
keep_cols = ["loan_id", "loan_issued_at", TARGET] + candidate_features
df_final = df[keep_cols].copy()
df_final.to_csv(r"E:\loan_dataset_engineered.csv", index=False)
print(f"Saved {len(df_final):,} rows, {len(candidate_features)} features to E:\\loan_dataset_engineered.csv")

Saved 306,470 rows, 8 features to E:\loan_dataset_engineered.csv


In [9]:
import pandas as pd

df = pd.read_csv(r"E:\loan_dataset_engineered.csv", parse_dates=["loan_issued_at"])
print(f"Loaded {len(df):,} rows")

TARGET = "has_default_within_12_months"

# =========================================================
# Split by year - this IS the time-based split
# =========================================================
df["issue_year"] = df["loan_issued_at"].dt.year

train = df[df["issue_year"].between(2018, 2021)].copy()
val = df[df["issue_year"] == 2022].copy()
test = df[df["issue_year"] == 2023].copy()

print(f"\nTrain (2018-2021): {len(train):,} rows")
print(f"Validation (2022): {len(val):,} rows")
print(f"Test (2023): {len(test):,} rows")

total_check = len(train) + len(val) + len(test)
print(f"\nSum check: {total_check:,} (should equal {len(df):,})")

# =========================================================
# Confirm each split's default rate is reasonable
# =========================================================
print("\nDefault rate by split:")
print(f"  Train:      {train[TARGET].mean():.2%}")
print(f"  Validation: {val[TARGET].mean():.2%}")
print(f"  Test:       {test[TARGET].mean():.2%}")

# =========================================================
# Separate features (X) from target (y)
# =========================================================
FEATURES = [
    "issued_amount", "initial_interest_rate", "initial_loan_duration",
    "loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
    "country_grouped", "customer_risk_rating_was_missing",
]

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"\nFeature matrix shapes:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")

# =========================================================
# Save all six pieces so the split is locked and
# reproducible - everyone on the team uses these exact files
# =========================================================
X_train.to_csv(r"E:\X_train.csv", index=False)
y_train.to_csv(r"E:\y_train.csv", index=False)
X_val.to_csv(r"E:\X_val.csv", index=False)
y_val.to_csv(r"E:\y_val.csv", index=False)
X_test.to_csv(r"E:\X_test.csv", index=False)
y_test.to_csv(r"E:\y_test.csv", index=False)

print("\nSaved 6 files to E:\\ — X_train, y_train, X_val, y_val, X_test, y_test")
print("This split is now LOCKED. Do not re-split randomly at any later stage.")

Loaded 306,470 rows

Train (2018-2021): 161,481 rows
Validation (2022): 63,594 rows
Test (2023): 81,395 rows

Sum check: 306,470 (should equal 306,470)

Default rate by split:
  Train:      20.84%
  Validation: 13.37%
  Test:       10.87%

Feature matrix shapes:
  X_train: (161481, 8), y_train: (161481,)
  X_val:   (63594, 8), y_val:   (63594,)
  X_test:  (81395, 8), y_test:  (81395,)

Saved 6 files to E:\ — X_train, y_train, X_val, y_val, X_test, y_test
This split is now LOCKED. Do not re-split randomly at any later stage.


In [10]:
!pip install joblib

In [11]:
""" Baseline Logistic Regression Model
Trains on X_train/y_train, evaluates on X_val/y_val.
This is the checkpoint deliverable - a working end-to-end model,
not necessarily the best one yet.
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

# =========================================================
# Load the locked split
# =========================================================
X_train = pd.read_csv(r"E:\X_train.csv")
y_train = pd.read_csv(r"E:\y_train.csv").squeeze()
X_val = pd.read_csv(r"E:\X_val.csv")
y_val = pd.read_csv(r"E:\y_val.csv").squeeze()

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

# =========================================================
# Identify column types for preprocessing
# =========================================================
numeric_cols = ["issued_amount", "initial_interest_rate", "initial_loan_duration",
                 "customer_risk_rating_was_missing"]
categorical_cols = ["loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
                     "country_grouped"]

# =========================================================
# Build a preprocessing + model pipeline
# (scaling for numeric, one-hot for categorical)
# =========================================================
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

# class_weight="balanced" addresses class imbalance (Section 4.3 of the plan)
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", model),
])

# =========================================================
# Train
# =========================================================
print("\nTraining Logistic Regression...")
pipeline.fit(X_train, y_train)
print("Done.")

# =========================================================
# Evaluate on validation set
# =========================================================
val_probs = pipeline.predict_proba(X_val)[:, 1]
val_preds = pipeline.predict(X_val)

roc_auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)

print(f"\n{'='*50}")
print("VALIDATION SET RESULTS (2022)")
print(f"{'='*50}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"\nConfusion matrix (threshold=0.5):")
print(confusion_matrix(y_val, val_preds))
print(f"\nClassification report:")
print(classification_report(y_val, val_preds))

# =========================================================
# Save the trained pipeline for later steps (calibration,
# SHAP, comparison against RF/XGBoost)
# =========================================================
import joblib
joblib.dump(pipeline, r"E:\baseline_logreg_model.pkl")
print("\nSaved trained pipeline to E:\\baseline_logreg_model.pkl")

X_train: (161481, 8), X_val: (63594, 8)

Training Logistic Regression...
Done.

VALIDATION SET RESULTS (2022)
ROC-AUC: 0.6600
PR-AUC:  0.2504

Confusion matrix (threshold=0.5):
[[52644  2448]
 [ 7001  1501]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.88      0.96      0.92     55092
         1.0       0.38      0.18      0.24      8502

    accuracy                           0.85     63594
   macro avg       0.63      0.57      0.58     63594
weighted avg       0.82      0.85      0.83     63594


Saved trained pipeline to E:\baseline_logreg_model.pkl


In [12]:
"""
Random Forest Model + Comparison vs Baseline
Trains on X_train/y_train, evaluates on X_val/y_val.
Directly compares against the Logistic Regression baseline.
"""

import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

# =========================================================
# Load the locked split (same files, same split, no changes)
# =========================================================
X_train = pd.read_csv(r"E:\X_train.csv")
y_train = pd.read_csv(r"E:\y_train.csv").squeeze()
X_val = pd.read_csv(r"E:\X_val.csv")
y_val = pd.read_csv(r"E:\y_val.csv").squeeze()

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

numeric_cols = ["issued_amount", "initial_interest_rate", "initial_loan_duration",
                 "customer_risk_rating_was_missing"]
categorical_cols = ["loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
                     "country_grouped"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

# class_weight="balanced" - same imbalance handling as baseline
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", rf_model),
])

print("\nTraining Random Forest...")
rf_pipeline.fit(X_train, y_train)
print("Done.")

# =========================================================
# Evaluate Random Forest on validation set
# =========================================================
rf_val_probs = rf_pipeline.predict_proba(X_val)[:, 1]
rf_val_preds = rf_pipeline.predict(X_val)

rf_roc_auc = roc_auc_score(y_val, rf_val_probs)
rf_pr_auc = average_precision_score(y_val, rf_val_probs)

print(f"\n{'='*50}")
print("RANDOM FOREST - VALIDATION RESULTS (2022)")
print(f"{'='*50}")
print(f"ROC-AUC: {rf_roc_auc:.4f}")
print(f"PR-AUC:  {rf_pr_auc:.4f}")
print(f"\nConfusion matrix (threshold=0.5):")
print(confusion_matrix(y_val, rf_val_preds))
print(f"\nClassification report:")
print(classification_report(y_val, rf_val_preds))

# =========================================================
# Direct comparison against the baseline
# =========================================================
baseline_pipeline = joblib.load(r"E:\baseline_logreg_model.pkl")
lr_val_probs = baseline_pipeline.predict_proba(X_val)[:, 1]
lr_roc_auc = roc_auc_score(y_val, lr_val_probs)
lr_pr_auc = average_precision_score(y_val, lr_val_probs)

print(f"\n{'='*50}")
print("COMPARISON: Logistic Regression vs Random Forest")
print(f"{'='*50}")
print(f"{'Metric':<12}{'LogReg':<12}{'RandomForest':<12}{'Difference':<12}")
print(f"{'ROC-AUC':<12}{lr_roc_auc:<12.4f}{rf_roc_auc:<12.4f}{rf_roc_auc-lr_roc_auc:+.4f}")
print(f"{'PR-AUC':<12}{lr_pr_auc:<12.4f}{rf_pr_auc:<12.4f}{rf_pr_auc-lr_pr_auc:+.4f}")

# =========================================================
# Feature importance (useful preview before the formal
# country-dominance check in Day 6-7)
# =========================================================
feature_names = (numeric_cols +
    list(rf_pipeline.named_steps["preprocess"]
         .named_transformers_["cat"]
         .get_feature_names_out(categorical_cols)))
importances = rf_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(f"\n{'='*50}")
print("TOP 10 FEATURE IMPORTANCES (Random Forest)")
print(f"{'='*50}")
print(importance_df.head(10).to_string(index=False))

# =========================================================
# Save the Random Forest pipeline
# =========================================================
joblib.dump(rf_pipeline, r"E:\random_forest_model.pkl")
print("\nSaved trained pipeline to E:\\random_forest_model.pkl")

X_train: (161481, 8), X_val: (63594, 8)

Training Random Forest...
Done.

RANDOM FOREST - VALIDATION RESULTS (2022)
ROC-AUC: 0.6753
PR-AUC:  0.2655

Confusion matrix (threshold=0.5):
[[52683  2409]
 [ 7020  1482]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.88      0.96      0.92     55092
         1.0       0.38      0.17      0.24      8502

    accuracy                           0.85     63594
   macro avg       0.63      0.57      0.58     63594
weighted avg       0.82      0.85      0.83     63594


COMPARISON: Logistic Regression vs Random Forest
Metric      LogReg      RandomForestDifference  
ROC-AUC     0.6600      0.6753      +0.0153
PR-AUC      0.2504      0.2655      +0.0151

TOP 10 FEATURE IMPORTANCES (Random Forest)
                       feature  importance
         initial_interest_rate    0.303499
interest_rate_bucket_very_high    0.159936
       country_grouped_Estonia    0.097933
 interest_rate_bucket_very_low  

In [15]:
!pip install xgboost

In [16]:
"""
XGBoost Model + Full Three-Way Comparison
Trains on X_train/y_train, evaluates on X_val/y_val.
Compares against both Logistic Regression and Random Forest.
"""

import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

# =========================================================
# Load the locked split (same files, no changes, ever)
# =========================================================
X_train = pd.read_csv(r"E:\X_train.csv")
y_train = pd.read_csv(r"E:\y_train.csv").squeeze()
X_val = pd.read_csv(r"E:\X_val.csv")
y_val = pd.read_csv(r"E:\y_val.csv").squeeze()

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

numeric_cols = ["issued_amount", "initial_interest_rate", "initial_loan_duration",
                 "customer_risk_rating_was_missing"]
categorical_cols = ["loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
                     "country_grouped"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

# scale_pos_weight handles class imbalance for XGBoost specifically
# (XGBoost doesn't use class_weight="balanced" like sklearn models)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"\nscale_pos_weight (for imbalance): {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", xgb_model),
])

print("\nTraining XGBoost...")
xgb_pipeline.fit(X_train, y_train)
print("Done.")

# =========================================================
# Evaluate XGBoost on validation set
# =========================================================
xgb_val_probs = xgb_pipeline.predict_proba(X_val)[:, 1]
xgb_val_preds = xgb_pipeline.predict(X_val)

xgb_roc_auc = roc_auc_score(y_val, xgb_val_probs)
xgb_pr_auc = average_precision_score(y_val, xgb_val_probs)

print(f"\n{'='*50}")
print("XGBOOST - VALIDATION RESULTS (2022)")
print(f"{'='*50}")
print(f"ROC-AUC: {xgb_roc_auc:.4f}")
print(f"PR-AUC:  {xgb_pr_auc:.4f}")
print(f"\nConfusion matrix (threshold=0.5):")
print(confusion_matrix(y_val, xgb_val_preds))
print(f"\nClassification report:")
print(classification_report(y_val, xgb_val_preds))

# =========================================================
# Full three-way comparison
# =========================================================
lr_pipeline = joblib.load(r"E:\baseline_logreg_model.pkl")
rf_pipeline = joblib.load(r"E:\random_forest_model.pkl")

lr_probs = lr_pipeline.predict_proba(X_val)[:, 1]
rf_probs = rf_pipeline.predict_proba(X_val)[:, 1]

lr_roc, lr_pr = roc_auc_score(y_val, lr_probs), average_precision_score(y_val, lr_probs)
rf_roc, rf_pr = roc_auc_score(y_val, rf_probs), average_precision_score(y_val, rf_probs)

print(f"\n{'='*50}")
print("THREE-WAY COMPARISON (Validation, 2022)")
print(f"{'='*50}")
print(f"{'Model':<20}{'ROC-AUC':<12}{'PR-AUC':<12}")
print(f"{'Logistic Regression':<20}{lr_roc:<12.4f}{lr_pr:<12.4f}")
print(f"{'Random Forest':<20}{rf_roc:<12.4f}{rf_pr:<12.4f}")
print(f"{'XGBoost':<20}{xgb_roc_auc:<12.4f}{xgb_pr_auc:<12.4f}")

best_model = max(
    [("Logistic Regression", lr_roc), ("Random Forest", rf_roc), ("XGBoost", xgb_roc_auc)],
    key=lambda x: x[1]
)
print(f"\nBest model by ROC-AUC: {best_model[0]} ({best_model[1]:.4f})")

# =========================================================
# XGBoost feature importance
# =========================================================
feature_names = (numeric_cols +
    list(xgb_pipeline.named_steps["preprocess"]
         .named_transformers_["cat"]
         .get_feature_names_out(categorical_cols)))
importances = xgb_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(f"\n{'='*50}")
print("TOP 10 FEATURE IMPORTANCES (XGBoost)")
print(f"{'='*50}")
print(importance_df.head(10).to_string(index=False))

# =========================================================
# Save the XGBoost pipeline
# =========================================================
joblib.dump(xgb_pipeline, r"E:\xgboost_model.pkl")
print("\nSaved trained pipeline to E:\\xgboost_model.pkl")
print("\nNEXT STEP: tuning (Day 8, tuning ONLY - do not finalize same day)")

X_train: (161481, 8), X_val: (63594, 8)

scale_pos_weight (for imbalance): 3.80

Training XGBoost...
Done.

XGBOOST - VALIDATION RESULTS (2022)
ROC-AUC: 0.6673
PR-AUC:  0.2479

Confusion matrix (threshold=0.5):
[[51762  3330]
 [ 6757  1745]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.88      0.94      0.91     55092
         1.0       0.34      0.21      0.26      8502

    accuracy                           0.84     63594
   macro avg       0.61      0.57      0.58     63594
weighted avg       0.81      0.84      0.82     63594


THREE-WAY COMPARISON (Validation, 2022)
Model               ROC-AUC     PR-AUC      
Logistic Regression 0.6600      0.2504      
Random Forest       0.6753      0.2655      
XGBoost             0.6673      0.2479      

Best model by ROC-AUC: Random Forest (0.6753)

TOP 10 FEATURE IMPORTANCES (XGBoost)
                 feature  importance
 country_grouped_Estonia    0.362157
   initial_interest_rate   

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

X_train = pd.read_csv(r"E:\X_train.csv")
y_train = pd.read_csv(r"E:\y_train.csv").squeeze()
X_val = pd.read_csv(r"E:\X_val.csv")
y_val = pd.read_csv(r"E:\y_val.csv").squeeze()

numeric_cols = ["issued_amount", "initial_interest_rate", "initial_loan_duration",
                 "customer_risk_rating_was_missing"]
categorical_cols_with_country = ["loan_amount_band", "loan_term_bucket",
                                   "interest_rate_bucket", "country_grouped"]
categorical_cols_no_country = ["loan_amount_band", "loan_term_bucket",
                                 "interest_rate_bucket"]

def build_and_evaluate(cat_cols, label):
    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    model = RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )
    pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    pipeline.fit(X_train[numeric_cols + cat_cols], y_train)

    val_probs = pipeline.predict_proba(X_val[numeric_cols + cat_cols])[:, 1]
    roc = roc_auc_score(y_val, val_probs)
    pr = average_precision_score(y_val, val_probs)
    print(f"{label}: ROC-AUC = {roc:.4f}, PR-AUC = {pr:.4f}")
    return roc, pr

print("Training WITH country_grouped...")
roc_with, pr_with = build_and_evaluate(categorical_cols_with_country, "WITH country")

print("\nTraining WITHOUT country_grouped...")
roc_without, pr_without = build_and_evaluate(categorical_cols_no_country, "WITHOUT country")

print(f"\n{'='*60}")
print("COUNTRY DOMINANCE CHECK - RESULTS")
print(f"{'='*60}")
print(f"{'Metric':<12}{'With Country':<16}{'Without Country':<18}{'Drop':<10}")
print(f"{'ROC-AUC':<12}{roc_with:<16.4f}{roc_without:<18.4f}{roc_with-roc_without:+.4f}")
print(f"{'PR-AUC':<12}{pr_with:<16.4f}{pr_without:<18.4f}{pr_with-pr_without:+.4f}")

drop_pct = (roc_with - roc_without) / roc_with * 100
print(f"\nROC-AUC drops by {drop_pct:.1f}% when country is removed.")

if drop_pct > 15:
    print("\nFINDING: Country carries a LARGE share of the model's predictive power.")
    print("This should be reported explicitly as a limitation - the model relies")
    print("heavily on geography, which may reflect underwriting/regulatory")
    print("differences by country rather than purely individual borrower risk.")
elif drop_pct > 5:
    print("\nFINDING: Country carries a MODERATE share of predictive power.")
    print("Worth noting in the report, not necessarily disqualifying.")
else:
    print("\nFINDING: Country's contribution is SMALL. Model relies mainly on")
    print("genuine borrower/loan-level features.")

Training WITH country_grouped...
WITH country: ROC-AUC = 0.6753, PR-AUC = 0.2655

Training WITHOUT country_grouped...
WITHOUT country: ROC-AUC = 0.6615, PR-AUC = 0.2351

COUNTRY DOMINANCE CHECK - RESULTS
Metric      With Country    Without Country   Drop      
ROC-AUC     0.6753          0.6615            +0.0138
PR-AUC      0.2655          0.2351            +0.0304

ROC-AUC drops by 2.0% when country is removed.

FINDING: Country's contribution is SMALL. Model relies mainly on
genuine borrower/loan-level features.


In [ ]:
"""
Hyperparameter Tuning (Random Forest, the winning model)

IMPORTANT: This is TUNING ONLY. Do not finalize the model today.
This script is deliberately time-boxed with a limited search (RandomizedSearchCV,
not exhaustive GridSearchCV) so it can't silently run for hours.

If this is still running after ~20-30 minutes, STOP IT (interrupt the kernel)
and use whatever best result was found so far - do not let tuning eat into
Day 8's calibration time. Per the plan: cut tuning depth before cutting
calibration.
"""

import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, make_scorer

X_train = pd.read_csv(r"E:\X_train.csv")
y_train = pd.read_csv(r"E:\y_train.csv").squeeze()
X_val = pd.read_csv(r"E:\X_val.csv")
y_val = pd.read_csv(r"E:\y_val.csv").squeeze()

numeric_cols = ["issued_amount", "initial_interest_rate", "initial_loan_duration",
                 "customer_risk_rating_was_missing"]
categorical_cols = ["loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
                     "country_grouped"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
])

# =========================================================
# Deliberately small, bounded search space - not exhaustive.
# This is a TIME-BOXED search, not a perfect one.
# =========================================================
param_distributions = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [6, 8, 10, 12, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
}

# n_iter=20 keeps this bounded - 20 combinations, not thousands
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="average_precision",  # optimizing for PR-AUC, the more honest metric here
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("Starting tuning (20 combinations, 3-fold CV = 60 fits total)...")
print("If this runs past ~20-30 minutes, interrupt the kernel and use the")
print("best result found so far - do not let this eat into Day 8's time.\n")

start_time = time.time()
search.fit(X_train, y_train)
elapsed = time.time() - start_time

print(f"\nTuning completed in {elapsed/60:.1f} minutes")
print(f"\nBest parameters found:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest cross-validation PR-AUC: {search.best_score_:.4f}")

# =========================================================
# Evaluate the tuned model on validation set
# =========================================================
best_model = search.best_estimator_
val_probs = best_model.predict_proba(X_val)[:, 1]

tuned_roc = roc_auc_score(y_val, val_probs)
tuned_pr = average_precision_score(y_val, val_probs)

print(f"\n{'='*50}")
print("TUNED MODEL - VALIDATION RESULTS (2022)")
print(f"{'='*50}")
print(f"ROC-AUC: {tuned_roc:.4f}  (untuned RF was 0.6753)")
print(f"PR-AUC:  {tuned_pr:.4f}  (untuned RF was 0.2655)")

improvement_roc = tuned_roc - 0.6753
improvement_pr = tuned_pr - 0.2655
print(f"\nImprovement vs untuned: ROC-AUC {improvement_roc:+.4f}, PR-AUC {improvement_pr:+.4f}")

if improvement_pr < 0.01:
    print("\nNOTE: Tuning gave a small improvement. This is normal and expected -")
    print("do not keep tuning further hoping for a big jump. Move to Day 8")
    print("(finalize + calibrate) with this result.")

# =========================================================
# Save the tuned model 
# =========================================================
import joblib
joblib.dump(best_model, r"E:\tuned_random_forest_model.pkl")
print("\nSaved tuned model to E:\\tuned_random_forest_model.pkl")
print("\nSTOP HERE FOR TODAY. Day 8 = finalize + calibrate this model.")
print("Do not finalize/calibrate in the same session - keep them separate")
print("per the plan, so debugging time isn't compressed.")

In [2]:
metadata = {
    "random_seed": RANDOM_SEED,
    "model_type": "RandomForestClassifier (tuned)",
    "train_period": "2018-2021",
    "validation_period": "2022",
    "test_period": "2023",
    "test_roc_auc": round(test_roc, 4),
    "test_pr_auc": round(test_pr, 4),
    "test_brier_score": round(test_brier, 4),
    "features": [
        "issued_amount", "initial_interest_rate", "initial_loan_duration",
        "loan_amount_band", "loan_term_bucket", "interest_rate_bucket",
        "country_grouped", "customer_risk_rating_was_missing",
    ],
    "calibration_applied": bool(max_gap > 0.05),  # <-- fixed: cast to Python bool
}

import json
with open(r"E:\model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved model_metadata.json")

import subprocess
result = subprocess.run(["pip", "freeze"], capture_output=True, text=True)
with open(r"E:\requirements.txt", "w") as f:
    f.write(result.stdout)
print("Saved requirements.txt")

Saved model_metadata.json
Saved requirements.txt


In [1]:
import joblib
import os

for f in ["FINAL_model.pkl", "model_metadata.json", "requirements.txt",
          "X_train.csv", "y_train.csv", "X_val.csv", "y_val.csv", "X_test.csv", "y_test.csv"]:
    path = f"E:\\{f}"
    exists = os.path.exists(path)
    print(f"{f}: {'FOUND' if exists else 'MISSING'}")

# Sanity load test
model = joblib.load(r"E:\FINAL_model.pkl")
import pandas as pd
X_val = pd.read_csv(r"E:\X_val.csv")
print("\nModel loads and predicts:", model.predict_proba(X_val.head(3))[:, 1])

FINAL_model.pkl: FOUND
model_metadata.json: FOUND
requirements.txt: FOUND
X_train.csv: FOUND
y_train.csv: FOUND
X_val.csv: FOUND
y_val.csv: FOUND
X_test.csv: FOUND
y_test.csv: FOUND

Model loads and predicts: [0.09012618 0.01762632 0.09293914]
